# EXP-019 — Pre-flight only (no feature extraction)

**Purpose:** verify the experiment scripts synthetically, then reproduce the weak-label class-count audit on the provided training reports.

**Safety:** this notebook deliberately does not run full feature extraction or train a model. If any target fails the pre-registered minimum of 20 weak positives and 20 weak negatives, the EXP-019 verdict must remain **WITHHELD** under the current protocol. Do not lower that floor after seeing counts.

In [1]:
from pathlib import Path
import os, sys, subprocess, json

REPO_URL = 'https://github.com/vixintry/rsna-knee-m1.git'
BRANCH = 'research/exp019-weak-labels'
REPO_DIR = Path('/kaggle/working/rsna-knee-m1')
DATA_DIR = Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    raise RuntimeError(f'{REPO_DIR} already exists. Remove it or use a fresh Kaggle session so the checked-out branch is unambiguous.')

assert (REPO_DIR / 'rsna-submission.ipynb').is_file(), 'Production notebook missing from clone.'
assert (REPO_DIR / 'scripts/exp019/extract_features.py').is_file(), 'EXP-019 extractor missing.'
assert (REPO_DIR / 'scripts/exp019/analyse_exp019.py').is_file(), 'EXP-019 analysis script missing.'
assert (REPO_DIR / 'scripts/derive_report_weak_labels.py').is_file(), 'Weak-label parser missing.'
assert (DATA_DIR / 'train.csv').is_file(), f'train.csv not found at {DATA_DIR}; check the competition data attachment.'
print('Repository:', REPO_DIR)
print('Branch:', BRANCH)
print('Competition train.csv:', DATA_DIR / 'train.csv')
print('Production notebook is present; it will not be edited.')

Cloning into '/kaggle/working/rsna-knee-m1'...


Repository: /kaggle/working/rsna-knee-m1
Branch: research/exp019-weak-labels
Competition train.csv: /kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv
Production notebook is present; it will not be edited.


## 1. Synthetic self-tests

These test code plumbing and fail-closed gates using synthetic data. They do **not** establish model quality or permit full extraction.

In [2]:
def run_checked(args):
    print('\n$ ' + ' '.join(map(str, args)))
    subprocess.run(args, cwd=REPO_DIR, check=True)

run_checked([sys.executable, 'scripts/exp019/extract_features.py', '--selftest'])
run_checked([sys.executable, 'scripts/exp019/analyse_exp019.py', '--selftest'])
print('\nBoth synthetic self-tests passed. Real DICOM/encoder extraction remains untested.')


$ /usr/bin/python3 scripts/exp019/extract_features.py --selftest
production-equivalence check passed: {"ranked_series": true, "ordered_dicom_paths": true, "dicom_to_image": true, "sample_indices": true, "N_SLICES": true, "SLICE_RANGE": true, "MAX_SERIES_TRIES": true}
production-equivalence check passed: {"ranked_series": true, "ordered_dicom_paths": true, "dicom_to_image": true, "sample_indices": true, "N_SLICES": true, "SLICE_RANGE": true, "MAX_SERIES_TRIES": true}
mode=full device=fake studies=7 (gold 2) out=/tmp/tmp6cuphiat/out
shard 1/3: 3 studies in 0s
shard 2/3: 3 studies in 0s
shard 3/3: 1 studies in 0s
series check: 2/2 gold studies checked | mismatches 0 | gold with rank>0 0 | passed=True (smoke runs can never pass)
{
  "mode": "full",
  "device": "fake",
  "elapsed_seconds": 0.3,
  "event_counts": {
    "stale_part_file_removed": 1,
    "series_failed": 2,
    "prod_study_failed": 1
  },
  "shards_present": 3,
  "shards_expected": 3
}
production-equivalence check passed: {"r

## 2. Generate weak labels locally

The committed deterministic parser reads the provided `train.csv` locally. Do not send report text to a hosted third-party API. Gold-labelled studies are excluded by the parser.

In [3]:
WEAK_DIR = Path('/kaggle/working/exp019_weak_labels')
PREFLIGHT_DIR = Path('/kaggle/working/exp019_preflight')
WEAK_DIR.mkdir(parents=True, exist_ok=True)
PREFLIGHT_DIR.mkdir(parents=True, exist_ok=True)

run_checked([
    sys.executable, 'scripts/derive_report_weak_labels.py',
    '--input-csv', str(DATA_DIR / 'train.csv'),
    '--output-dir', str(WEAK_DIR),
])
WEAK_CSV = WEAK_DIR / 'weak_labels.csv'
AUDIT_CSV = WEAK_DIR / 'weak_label_audit.csv'
assert WEAK_CSV.is_file() and AUDIT_CSV.is_file(), 'Parser did not create expected outputs.'
print('Weak labels:', WEAK_CSV)
print('Per-target cue audit:', AUDIT_CSV)


$ /usr/bin/python3 scripts/derive_report_weak_labels.py --input-csv /kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv --output-dir /kaggle/working/exp019_weak_labels
Total studies: 4407
Studies with official labels excluded: 58
Unlabelled studies to process: 4349

Weak-label coverage by target:
                  known  positive  negative  unknown  uncertain  conflict
target                                                                   
ACL                 971       370       601     3342         33         3
Baker's            1029       524       505     3312          3         5
Contusion           624       225       399     3713          9         3
Effusion           1948      1302       646     2391          2         8
Fracture            667       171       496     3655         18         9
Lateral Meniscus   1019       258       761     3281         41         8
Lateral OA           24        24         0     4325          0         0
MCL               

## 3. Registered class-count gate (CPU only)

Every target must have at least 20 weak positives **and** 20 weak negatives after removing gold studies. If any target fails, stop here. Do not run `extract_features.py` without first resolving the blocker under a new, pre-registered decision.

In [4]:
audit_cmd = [
    sys.executable, 'scripts/exp019/analyse_exp019.py',
    '--audit-weak-labels',
    '--weak-labels', str(WEAK_CSV),
    '--train-csv', str(DATA_DIR / 'train.csv'),
    '--out', str(PREFLIGHT_DIR),
]
print('\n$ ' + ' '.join(map(str, audit_cmd)))
audit_run = subprocess.run(audit_cmd, cwd=REPO_DIR, text=True, capture_output=True)
print(audit_run.stdout)
if audit_run.stderr:
    print('STDERR:', audit_run.stderr)
audit_path = PREFLIGHT_DIR / 'weak_label_audit.json'
assert audit_path.is_file(), 'Expected audit JSON was not written.'
audit = json.loads(audit_path.read_text())
print('\nAll 12 targets meet the registered floor:', audit['all_12_can_be_modelled'])
print('Targets below floor:', audit['targets_below_floor'])
if not audit['all_12_can_be_modelled']:
    print('\nSTOP: EXP-019 cannot issue its registered KEEP/INVESTIGATE/REJECT verdict under the current protocol. No feature extraction or model training was run.')
else:
    print('\nPre-flight class-count gate passed only. The separate rules/quality gate is still required before any full extraction.')
print('Audit artifact:', audit_path)


$ /usr/bin/python3 scripts/exp019/analyse_exp019.py --audit-weak-labels --weak-labels /kaggle/working/exp019_weak_labels/weak_labels.csv --train-csv /kaggle/input/competitions/rsna-knee-abnormality-detection/train.csv --out /kaggle/working/exp019_preflight
weak-label audit: 4349 studies (0 gold rows removed); floor = 20 positives AND negatives per target
  ACL               pos   370 | neg   601 | no call  3378 | OK
  MCL               pos   212 | neg   676 | no call  3461 | OK
  Medial Meniscus   pos   677 | neg   424 | no call  3248 | OK
  Lateral Meniscus  pos   258 | neg   761 | no call  3330 | OK
  Medial OA         pos    55 | neg     0 | no call  4294 | BELOW FLOOR -> not modelled
  Lateral OA        pos    24 | neg     0 | no call  4325 | BELOW FLOOR -> not modelled
  PF OA             pos    70 | neg     0 | no call  4279 | BELOW FLOOR -> not modelled
  Effusion          pos  1302 | neg   646 | no call  2401 | OK
  Synovitis         pos   381 | neg    12 | no call  3956 | BEL

## 4. Human gate still required

Do **not** set `status` to `VERIFIED` automatically. Before full extraction, a human must read the full official competition rules (including data-security terms), confirm report-derived labels are allowed under those terms, verify the metric on the official evaluation page, and complete `scripts/exp019/exp019_gates.json`. The weak-label quality gate must also be satisfied or explicitly waived in `DECISIONS.md`.

This notebook intentionally contains no full-extraction command. The production notebook and submission pipeline are not modified.

In [ ]:

## EXP-020 — Local review sampling results

**Status:** Sampling completed; human review pending.

**Configuration**
- Random seed: 20261020
- Requested sample per stratum: 20
- Input studies: 4,407
- Studies with any official label excluded: 58
- Unlabelled studies considered: 4,349
- Review rows generated: 182

**Coverage findings**
- Positive and unknown strata were sampled for all four targets.
- Medial OA uncertain: 1 row.
- Synovitis conflict: 1 row.
- Synovitis negative: 12 rows.
- Synovitis uncertain: 8 rows.
- Eight of the 20 possible target/status combinations were absent from the summary, consistent with zero observed candidates in those strata.

**Provenance**
- Input SHA-256: `8ca2203c0e9d61c080c7a314c7cdb51c1b03a1d9eb4770819f7f34af53ef4e33`
- Parser SHA-256: `d19623223cc146687b9665afd240bcc073ece06467584dae3161f25322b6bd72`
- Output directory: `/kaggle/working/exp020_review/`

**Limitations**
- Parser accuracy has not yet been established.
- No training or model evaluation was performed.
- Report text remains local only; do not commit or upload it.
- No parser rules or registered experiment gates were changed.

**Next action:** Conduct human review using the predefined reference-label categories, then evaluate parser agreement without treating unknown reports as negative.


In [ ]:

## EXP-020 — Human-review protocol

### Objective
Evaluate whether the report-derived weak-label parser correctly identifies textual evidence for Medial OA, Lateral OA, PF OA and Synovitis.

Human annotations are reference judgements about the report text, not independently verified clinical diagnoses.

### Annotation procedure
1. Review each sampled report in its full available context.
2. Assign a reference label independently of the parser's predicted status wherever practical.
3. Use the definitions below consistently.
4. Record a concise explanation in reviewer_notes when the wording is ambiguous or a decision is difficult.
5. Do not modify parser rules while annotation is in progress.

### Reference-label definitions
- `positive`: Explicit evidence that the target abnormality is present in the report's findings or impression.
- `negative`: Explicit evidence that the target abnormality is absent or normal.
- `uncertain`: The report explicitly expresses doubt, suspicion or an unresolved possibility about the target.
- `not_stated`: The report does not provide sufficient evidence to assign one of the preceding categories.

### Interpretation rules
- Do not infer a negative label from silence.
- Distinguish clinical history or indication from findings and impression.
- Consider negation, uncertainty, anatomical compartment and the scope of each statement.
- Do not treat a mention of a different compartment or abnormality as evidence for the target.
- If positive and negative evidence conflict, assess the full report context and document the ambiguity.
- If a report genuinely cannot be interpreted, flag it in reviewer_notes rather than guessing.

### Quality control
- Keep the parser's original output unchanged.
- Preserve the existing reference_label and reviewer_notes columns.
- If a second reviewer is available, independently annotate a subset and compare disagreements.
- Resolve disagreements through documented adjudication where feasible.
- Report the number of reviewed rows, category counts, missing annotations and disagreements.

### Planned evaluation
Compare parser statuses against the human reference labels, separately for each target.

Report:
- Number of annotated rows and reference-label distribution.
- Coverage: proportion of rows for which the parser produced a positive or negative label.
- Agreement on rows with determinate positive/negative reference labels.
- False-positive and false-negative counts on those determinate rows.
- Uncertain, not-stated and conflict cases separately.

Do not interpret unknown parser outputs as negative predictions. Do not claim clinical validity from report-text agreement alone.

### Data handling
- Review the local-only CSV at `/kaggle/working/exp020_review/exp020_review_sample_LOCAL_ONLY.csv`.
- Never commit, upload or paste report text into chat or hosted services.
- Record aggregate counts and evaluation results only.
- No model training is part of this experiment.

### Status
Protocol drafted; human annotation and parser evaluation pending.


In [10]:
from pathlib import Path
import pandas as pd

review_path = Path(
    "/kaggle/working/exp020_review/"
    "exp020_review_sample_LOCAL_ONLY.csv"
)

assert review_path.exists(), f"Missing review CSV: {review_path}"

review = pd.read_csv(review_path)

print("Rows:", len(review))
print("Columns:", review.columns.tolist())
print("\nTarget counts:")
print(review["target"].value_counts().sort_index().to_string())

print("\nParser status counts:")
print(review["parser_status"].value_counts().sort_index().to_string())

required = {
    "review_row_id",
    "StudyInstanceUID",
    "target",
    "parser_status",
    "parser_label",
    "report_text_LOCAL_ONLY",
    "reference_label",
    "reviewer_notes",
}

print("\nMissing required columns:", sorted(required - set(review.columns)))
print("Duplicate review IDs:", review["review_row_id"].duplicated().sum())
print("Unannotated reference labels:", review["reference_label"].isna().sum())

Rows: 182
Columns: ['review_row_id', 'StudyInstanceUID', 'target', 'parser_status', 'parser_label', 'positive_cues', 'negative_cues', 'report_text_LOCAL_ONLY', 'reference_label', 'reviewer_notes']

Target counts:
target
Lateral OA    40
Medial OA     41
PF OA         40
Synovitis     61

Parser status counts:
parser_status
conflict      1
negative     12
positive     80
uncertain     9
unknown      80

Missing required columns: []
Duplicate review IDs: 0
Unannotated reference labels: 182


In [12]:

from pathlib import Path
import html
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

review_dir = Path("/kaggle/working/exp020_review")
annotation_path = (
    review_dir / "exp020_review_annotations_LOCAL_ONLY.csv"
)

assert annotation_path.exists(), "Annotation copy is missing."

df = pd.read_csv(annotation_path, keep_default_na=False)

required = {
    "review_row_id", "target", "report_text_LOCAL_ONLY",
    "reference_label", "reviewer_notes"
}
assert required.issubset(df.columns)
assert df["review_row_id"].is_unique

labels = ["", "positive", "negative", "uncertain", "not_stated"]
state = {"index": 0}

# Resume at the first row that still needs a reference label.
pending = df.index[df["reference_label"].eq("")].tolist()
if pending:
    state["index"] = pending[0]

output = widgets.Output()
progress = widgets.HTML()
report_box = widgets.HTML()
label_box = widgets.Dropdown(
    options=[(x if x else "-- Select a label --", x) for x in labels],
    description="Reference:",
    value="",
    layout=widgets.Layout(width="350px")
)
notes_box = widgets.Textarea(
    description="Notes:",
    placeholder="Optional: explain ambiguity or difficult decisions",
    layout=widgets.Layout(width="100%", height="100px")
)
save_button = widgets.Button(description="Save annotation",
                             button_style="primary")
prev_button = widgets.Button(description="Previous")
next_button = widgets.Button(description="Next")
message = widgets.HTML()

def save_current():
    i = state["index"]
    label = label_box.value

    if label not in labels[1:]:
        message.value = (
            "<span style='color:#b45309'>"
            "Choose a reference label before saving.</span>"
        )
        return False

    df.at[i, "reference_label"] = label
    df.at[i, "reviewer_notes"] = notes_box.value

    # Save the complete annotation copy, preserving original parser fields.
    df.to_csv(annotation_path, index=False)
    message.value = (
        "<span style='color:green'>Saved locally.</span>"
    )
    return True

def render():
    i = state["index"]
    row = df.iloc[i]

    progress.value = (
        f"<b>Review {i + 1} of {len(df)}</b> · "
        f"Target: {html.escape(str(row['target']))} · "
        f"ID: {html.escape(str(row['review_row_id']))}"
    )

    report_box.value = (
        "<div style='white-space:pre-wrap; overflow-wrap:anywhere; "
        "max-height:420px; overflow-y:auto; border:1px solid #888; "
        "padding:12px; border-radius:6px'>"
        + html.escape(str(row["report_text_LOCAL_ONLY"]))
        + "</div>"
    )

    current_label = str(row["reference_label"])
    label_box.value = current_label if current_label in labels else ""
    notes_box.value = str(row["reviewer_notes"])
    message.value = ""

def on_save(_):
    if save_current():
        render()

def on_prev(_):
    # Save edits before navigating if a label has been selected.
    if label_box.value:
        if not save_current():
            return
    elif label_box.value == "":
        current_notes = notes_box.value
        if current_notes.strip():
            message.value = (
                "<span style='color:#b45309'>"
                "Select a label and save, or clear unsaved notes "
                "before navigating.</span>"
            )
            return

    if state["index"] > 0:
        state["index"] -= 1
        render()

def on_next(_):
    if not label_box.value:
        message.value = (
            "<span style='color:#b45309'>"
            "Select a label before moving on.</span>"
        )
        return

    if not save_current():
        return

    if state["index"] < len(df) - 1:
        state["index"] += 1
        render()
    else:
        message.value = (
            "<b>End of sample.</b> Your final annotation was saved."
        )

save_button.on_click(on_save)
prev_button.on_click(on_prev)
next_button.on_click(on_next)

render()

display(progress, report_box, label_box, notes_box,
        widgets.HBox([prev_button, save_button, next_button]),
        message)


HTML(value='<b>Review 1 of 182</b> · Target: Lateral OA · ID: R00001')

HTML(value="<div style='white-space:pre-wrap; overflow-wrap:anywhere; max-height:420px; overflow-y:auto; borde…

Dropdown(description='Reference:', layout=Layout(width='350px'), options=(('-- Select a label --', ''), ('posi…

Textarea(value='', description='Notes:', layout=Layout(height='100px', width='100%'), placeholder='Optional: e…

HTML(value='')